# Mission 01: E-Commerce Analytics Warehouse

The Chinook music store wants a reporting warehouse. Analysts must answer revenue, retention and product questions **without** running heavy queries on the application database.

**Your job:** design a star schema in the lab database, load it from Chinook, and write the reporting queries.

**Lessons used:** 02 (keys), 03 (DDL), 05 (joins), 06 (CTEs), 07 (window functions), 10 (indexes), 15 (star schemas).

## Setup

Run this cell first. It starts PostgreSQL in Docker, loads Chinook, and gives you two helpers:

- `q(sql)` reads from the source database `chinook`.
- `run(sql)` executes statements and `lq(sql)` reads in the lab database `course`, where you build the warehouse.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine, text

sys.path.insert(0, str(Path.cwd().parent.parent / "tools"))
import datasets                                   # course helper: downloads and loads the sample data

engine = create_engine(datasets.ensure_chinook())  # starts PostgreSQL in Docker and loads Chinook on first use

def q(sql: str, **params) -> pd.DataFrame:
    """Run a SQL query and return the result as a DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

lab = create_engine(datasets.POSTGRES_URL.format(db="course"))   # a separate database for experiments

def run(sql: str) -> None:
    """Execute SQL statements (DDL/DML) in the lab database, in one transaction."""
    with lab.begin() as conn:
        if "$$" in sql:                       # function bodies contain ";" -> send the whole script at once
            conn.execute(text(sql))
            return
        for stmt in [s for s in sql.split(";") if s.strip()]:
            conn.execute(text(stmt))

def lq(sql: str, **params) -> pd.DataFrame:
    with lab.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

REGION = {
    "USA": "North America", "Canada": "North America",
    "Brazil": "South America", "Argentina": "South America", "Chile": "South America",
    "India": "Asia-Pacific", "Australia": "Asia-Pacific",
}   # every other Chinook country is in Europe

## Tasks

1. **Schema.** In a schema `mart`, create `dim_customer` (with a `region` column from the `REGION` mapping), `dim_product` (track, album, artist, genre), `dim_date` and `fact_sales` (grain: one invoice line). Use surrogate keys, primary keys and foreign keys.
2. **Load.** Fill the tables from Chinook. Check that the total revenue in `fact_sales` equals `sum(total)` of the source `invoice` table.
3. **Monthly revenue.** Return monthly revenue per region, plus a running total per region (window function). Plot it.
4. **Retention.** For each cohort (year of first purchase), compute the share of customers who bought again within 90 days of their first purchase.
5. **Top products.** Return the top 3 genres by revenue in each region (`DENSE_RANK`).
6. **Business question.** Answer in plain English: "Which region grew the most from 2024 to 2025, in absolute and relative terms?"
7. **Performance.** Propose one index for your most frequent query, create it, and show with `EXPLAIN` that PostgreSQL can use it.

In [ ]:
# Task 1: schema
run("""
DROP SCHEMA IF EXISTS mart CASCADE;
CREATE SCHEMA mart;
-- CREATE TABLE mart.dim_customer (...);
""")

In [ ]:
# Task 2: load and reconcile
...

In [ ]:
# Task 3: monthly revenue per region with a running total
...

In [ ]:
# Task 4: 90-day repeat-purchase rate per cohort
...

In [ ]:
# Task 5: top 3 genres per region
...

In [ ]:
# Task 6: growth 2024 -> 2025 per region
...

In [ ]:
# Task 7: index + EXPLAIN
...

## Submission Checklist

- [ ] The fact table has a clear grain, and its revenue reconciles with the source.
- [ ] Every query runs on the `mart` schema, not on the source tables.
- [ ] At least one window function and one CTE.
- [ ] One plain-English answer with numbers.
- [ ] One index, justified and checked with `EXPLAIN`.

A solution is in `01-E-Commerce-Analytics-Warehouse-solution.ipynb`. Try the mission first.